<a href="https://colab.research.google.com/github/OscarPellicer/extra-attributions/blob/main/notebooks/02_dl_attributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 · Hands-on: explaining a CNN

We explain the predictions of an ImageNet CNN (**VGG16**) with [Captum](https://captum.ai): gradient-based methods, Integrated Gradients, Grad-CAM, a **contrastive** explanation (*why hen and not rooster?*) and a **sanity check** (Adebayo et al., 2018).

Run the cells in order. Parameters marked `# change me` list values worth trying: change one, re-run the cell and compare. The first run downloads the VGG16 weights (~530 MB).

*A longer version (SmoothGrad, Occlusion, LRP, Quantus metrics, extras, with exercises and answers) is in `extended/02_dl_attributions.ipynb`.*

In [ ]:
# Install the dependencies (only on Google Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q captum quantus lovely-tensors git+https://github.com/OscarPellicer/extra-attributions.git
else:
    print('Running locally. If anything is missing, from the root of the extra-attributions repo run:\n'
          '    pip install -r requirements.txt\n    pip install -e .')

In [ ]:
import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')  # Apple-silicon: fall back to CPU for unsupported ops

import copy, json, time, urllib.request, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

from captum.attr import (Saliency, GuidedBackprop, InputXGradient, IntegratedGradients, NoiseTunnel,
                         LayerGradCam, GuidedGradCam, LayerAttribution, Occlusion, LRP)
from captum.attr._utils.lrp_rules import EpsilonRule, GammaRule

import lovely_tensors as lt
lt.monkey_patch()  # nicer printing of tensors
warnings.filterwarnings('ignore', message='.*required_grads has been set automatically.*')
warnings.filterwarnings('ignore', message='.*Setting backward hooks on ReLU activations.*')

# Device: CUDA > Apple MPS > CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f'Using {DEVICE=} | torch {torch.__version__}')
torch.manual_seed(0); np.random.seed(0)

Helpers: image loading and plotting. **Green** = evidence *for* the class, **red** = *against*.

In [ ]:
MEDIA_URL = 'https://raw.githubusercontent.com/OscarPellicer/extra-attributions/main/notebooks/media/'

def get_media(fname):
    '''Return a local path to `fname`, downloading it into ./media if needed.'''
    for folder in [Path('media'), Path('../media'), Path('notebooks/media')]:
        if (folder / fname).exists():
            return folder / fname
    Path('media').mkdir(exist_ok=True)
    urllib.request.urlretrieve(MEDIA_URL + fname, Path('media') / fname)
    return Path('media') / fname

# Red-white-green colour map for signed attributions
_colors, _values = ['darkred', 'red', 'lightcoral', 'white', 'palegreen', 'green', 'darkgreen'], [0, .15, .4, .5, .6, .9, 1.]
RdGn = LinearSegmentedColormap.from_list('RdGn', list(zip(_values, _colors)), N=256)

def to_2d(attr):
    '''(1,C,H,W) / (C,H,W) / (H,W) attribution -> (H,W) numpy array (mean over channels).'''
    a = torch.as_tensor(attr).detach().float().cpu()
    while a.dim() > 3:
        a = a[0]
    return (a.mean(0) if a.dim() == 3 else a).numpy()

def to_rgb(image):
    return image.detach().cpu().permute(1, 2, 0).clamp(0, 1).numpy()

def plot_attributions(image, attributions=[], attribution_names=[], limits=None, title=None,
                      figsize=3, percentile=99.5):
    '''Plot the input image followed by one signed attribution map per method.'''
    n = 1 + len(attributions)
    fig, axes = plt.subplots(1, n, figsize=(figsize * n, figsize + 0.5), squeeze=False)
    axes = axes[0]
    axes[0].imshow(to_rgb(image)); axes[0].set_title('Input')
    for i, (ax, attr, name) in enumerate(zip(axes[1:], attributions, attribution_names)):
        m = to_2d(attr)
        lim = limits[i] if isinstance(limits, list) else limits
        if lim is None:
            v = max(np.percentile(np.abs(m), percentile), 1e-12)
            lim = (-v, v)
        im = ax.imshow(m, cmap=RdGn, vmin=lim[0], vmax=lim[1])
        ax.set_title(name)
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03); cb.ax.tick_params(labelsize=7)
    for ax in axes:
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout(); plt.show()

def plot_overlay(image, heatmaps, names, title=None, figsize=3):
    '''Classic Grad-CAM style: non-negative heatmaps overlaid on the image.'''
    n = 1 + len(heatmaps)
    fig, axes = plt.subplots(1, n, figsize=(figsize * n, figsize + 0.5), squeeze=False)
    axes = axes[0]
    axes[0].imshow(to_rgb(image)); axes[0].set_title('Input')
    for ax, h, name in zip(axes[1:], heatmaps, names):
        h = to_2d(h)
        ax.imshow(to_rgb(image)); ax.imshow(h / (h.max() + 1e-12), cmap='jet', alpha=0.5, vmin=0, vmax=1)
        ax.set_title(name)
    for ax in axes:
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout(); plt.show()

In [ ]:
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).eval().to(DEVICE)
for m in model.modules():
    if isinstance(m, nn.ReLU):
        m.inplace = False

with open(get_media('imagenet_class_index.json')) as f:
    idx_to_labels = json.load(f)
def idx_to_label(i):
    return idx_to_labels[str(int(i))][1]
label_to_idx = {v[1]: int(k) for k, v in idx_to_labels.items()}

# We resize the whole image to 224x224 (instead of a center crop) so that nothing is cut out
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def load_image(fname, crop=None):
    '''Returns (image in [0,1] (3,224,224) on CPU, normalised input batch (1,3,224,224) on DEVICE).'''
    img = Image.open(get_media(fname)).convert('RGB')
    if crop is not None:
        img = img.crop(crop)
    x_img = transform(img)
    return x_img, normalize(x_img)[None].to(DEVICE)

def show_top5(x):
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)
    top_p, top_i = probs.topk(5)
    for k, (p, i) in enumerate(zip(top_p[0], top_i[0])):
        print(f' {k+1}. {idx_to_label(i):<20s} {p.item()*100:6.2f}%   (class index {i.item()})')
    return top_i[0].tolist()

In [ ]:
IMAGE = 'hen_rooster.jpg'   # change me: 'hen_rooster.jpg' or 'swan.jpg'

x_img, x = load_image(IMAGE)
print(x)  # lovely_tensors summary of the normalised input
top5 = show_top5(x)
pred_idx = top5[0]
plt.figure(figsize=(3, 3)); plt.imshow(to_rgb(x_img)); plt.axis('off'); plt.title(IMAGE); plt.show()

The model hesitates between **cock** and **hen**: both birds are in the picture.

## 1. Gradient-based methods

* **Saliency**: $\partial f_c / \partial x$, which pixels change the class score most.
* **Guided Backprop**: gradient with negative values zeroed at every ReLU. Very crisp maps... (does it change with the class?)
* **Input×Gradient**: $x \odot \partial f_c / \partial x$.

In [ ]:
TARGET = pred_idx   # change me: e.g. label_to_idx['hen'], label_to_idx['cock'], label_to_idx['goldfish']

attr_sal = Saliency(model).attribute(x, target=TARGET, abs=False)
attr_gbp = GuidedBackprop(model).attribute(x, target=TARGET)
attr_ixg = InputXGradient(model).attribute(x, target=TARGET)

plot_attributions(x_img, [attr_sal, attr_gbp, attr_ixg], ['Saliency', 'Guided Backprop', 'Input×Gradient'],
                  title=f'Target: {idx_to_label(TARGET)}')

## 2. Integrated Gradients

IG integrates the gradient along a path from a **baseline** $x'$ to the input, and its attributions add up to $f_c(x) - f_c(x')$ (`delta` = error of the numerical integral). The baseline means "absence of signal": with a **black** baseline, black pixels (the rooster's body, the background) always get zero attribution.

In [ ]:
def make_baseline(kind, x_img):
    '''Baselines are built in [0,1] image space and then normalised like the input.'''
    if kind == 'black':
        b = torch.zeros_like(x_img)
    elif kind == 'white':
        b = torch.ones_like(x_img)
    elif kind == 'mean':      # = 0 in normalised space
        b = torch.tensor([0.485, 0.456, 0.406])[:, None, None].expand_as(x_img)
    elif kind == 'blur':
        b = transforms.GaussianBlur(kernel_size=51, sigma=20)(x_img)
    elif kind == 'uniform noise':
        b = torch.rand(x_img.shape, generator=torch.Generator().manual_seed(0))
    else:
        raise ValueError(kind)
    return normalize(b)[None].to(DEVICE)

In [ ]:
TARGET = pred_idx                                  # change me
N_STEPS = 32                                       # change me: e.g. 2, 8, 32, 128
BASELINES = ['black', 'mean', 'blur', 'uniform noise']   # change me: also 'white'

ig = IntegratedGradients(model)
attr_ig_all = {}
for kind in BASELINES:
    baseline = make_baseline(kind, x_img)
    attr, delta = ig.attribute(x, baselines=baseline, target=TARGET, n_steps=N_STEPS,
                               internal_batch_size=16, return_convergence_delta=True)
    with torch.no_grad():
        diff = (model(x)[0, TARGET] - model(baseline)[0, TARGET]).item()
    print(f'{kind:>14s} baseline | sum(IG) = {attr.sum().item():7.3f} | f(x) - f(baseline) = {diff:7.3f} '
          f'| delta = {delta.item():+.4f}')
    attr_ig_all[kind] = attr
attr_ig = attr_ig_all[BASELINES[0]]

plot_attributions(x_img, list(attr_ig_all.values()), [f'IG ({k} baseline)' for k in attr_ig_all],
                  title=f'Target: {idx_to_label(TARGET)}  |  n_steps = {N_STEPS}')

## 3. Grad-CAM

Grad-CAM weights the channels of a convolutional layer by their average gradient. The map has the resolution of that layer (14×14 for layer 29, upsampled). Is it really class-discriminative? Compare *cock* and *hen*:

In [ ]:
TARGET = pred_idx      # change me
GRADCAM_LAYER = 29     # change me: index in model.features, e.g. 30 (7x7), 29 (14x14), 22 (28x28), 15 (56x56)

layer = model.features[GRADCAM_LAYER]
attr_gc = LayerGradCam(model, layer).attribute(x, target=TARGET, relu_attributions=True)
attr_gc_up = LayerAttribution.interpolate(attr_gc, x.shape[-2:], interpolate_mode='bilinear')
attr_ggc = GuidedGradCam(model, layer).attribute(x, target=TARGET)
print(f'Layer: model.features[{GRADCAM_LAYER}] = {layer} | Grad-CAM resolution: {tuple(attr_gc.shape[-2:])}')

plot_attributions(x_img, [attr_gc, attr_gc_up, attr_ggc], ['Grad-CAM (raw)', 'Grad-CAM (upsampled)', 'Guided Grad-CAM'],
                  title=f'Target: {idx_to_label(TARGET)}')

In [ ]:
heatmaps = []
for c in top5[:2]:
    gc = LayerGradCam(model, model.features[GRADCAM_LAYER]).attribute(x, target=c, relu_attributions=True)
    heatmaps.append(LayerAttribution.interpolate(gc, x.shape[-2:], interpolate_mode='bilinear'))
plot_overlay(x_img, heatmaps, [f'Grad-CAM: {idx_to_label(c)}' for c in top5[:2]])

## 4. Contrastive explanation: *why hen and not rooster?*

Explaining $f_\text{hen}(x)$ alone mixes "it is a chicken" (shared by both classes) with "it is a hen rather than a rooster". Explaining the **difference of logits** $f_A(x) - f_B(x)$ removes the shared evidence.

In [ ]:
class LogitDifference(nn.Module):
    '''Wraps a classifier so that its single output is logit[class_a] - logit[class_b].'''
    def __init__(self, model, class_a, class_b):
        super().__init__()
        self.model, self.class_a, self.class_b = model, class_a, class_b
    def forward(self, x):
        logits = self.model(x)
        return logits[:, [self.class_a]] - logits[:, [self.class_b]]

In [ ]:
CLASS_A = 'hen'                       # change me
CLASS_B = 'cock'                      # change me: e.g. 'cock', 'partridge', 'goldfish'
CONTRAST_IMAGE = 'hen_rooster.jpg'    # change me

xc_img, xc = load_image(CONTRAST_IMAGE)
a, b = label_to_idx[CLASS_A], label_to_idx[CLASS_B]
with torch.no_grad():
    logits = model(xc)[0]
print(f'logit[{CLASS_A}] = {logits[a]:.2f}, logit[{CLASS_B}] = {logits[b]:.2f}, difference = {logits[a] - logits[b]:.2f}')

models_to_explain = {CLASS_A: (model, a), CLASS_B: (model, b),
                     f'{CLASS_A} − {CLASS_B}': (LogitDifference(model, a, b), 0),
                     f'{CLASS_B} − {CLASS_A}': (LogitDifference(model, b, a), 0)}
occ_maps, gc_maps = {}, {}
for name, (f, t) in models_to_explain.items():
    gc = LayerGradCam(f, model.features[29]).attribute(xc, target=t, relu_attributions=True)
    gc_maps[name] = LayerAttribution.interpolate(gc, xc.shape[-2:], interpolate_mode='bilinear')
    occ_maps[name] = Occlusion(f).attribute(xc, target=t, sliding_window_shapes=(3, 32, 32), strides=(3, 16, 16),
                                            baselines=0, perturbations_per_eval=32)

names = list(models_to_explain)
plot_overlay(xc_img, [gc_maps[n] for n in names], [f'Grad-CAM: {n}' for n in names], title='Grad-CAM')
plot_attributions(xc_img, [occ_maps[n] for n in names[:3]], [f'Occlusion: {n}' for n in names[:3]],
                  title='Occlusion (w=32, s=16)')

## 5. Sanity check: randomise the model

If an explanation is faithful to the model, it must **change when the model is destroyed**. We re-initialise the weights layer by layer from the output to the input and compare each map with the original one (Spearman correlation). Which method keeps showing the birds when the network is random?

In [ ]:
SANITY_METHODS = ['Saliency', 'Guided Backprop', 'Integrated Gradients', 'Grad-CAM']

def explain(method, model, x, target):
    '''Common interface used in the sanity check and in the Quantus evaluation. Returns an (N,C,H,W) tensor.'''
    if method == 'Saliency':
        return Saliency(model).attribute(x, target=target, abs=False)
    if method == 'Guided Backprop':
        return GuidedBackprop(model).attribute(x, target=target)
    if method == 'Input×Gradient':
        return InputXGradient(model).attribute(x, target=target)
    if method == 'Integrated Gradients':
        return IntegratedGradients(model).attribute(x, target=target, n_steps=16, internal_batch_size=16)
    if method == 'SmoothGrad':
        return NoiseTunnel(Saliency(model)).attribute(x, nt_type='smoothgrad', nt_samples=8, nt_samples_batch_size=8,
                                                      stdevs=0.15 * (x.max() - x.min()).item(), target=target, abs=False)
    if method == 'Grad-CAM':
        gc = LayerGradCam(model, model.features[29]).attribute(x, target=target, relu_attributions=True)
        return LayerAttribution.interpolate(gc, x.shape[-2:], interpolate_mode='bilinear')
    raise ValueError(method)

# Names of the 16 weight layers of VGG16, from input to output
vgg_layers = [name for name, m in model.named_modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
block_sizes = [2, 2, 3, 3, 3]
pretty = [f'conv{b+1}_{i+1}' for b, n in enumerate(block_sizes) for i in range(n)] + ['fc6', 'fc7', 'fc8']
layer_names = dict(zip(vgg_layers, pretty))
print(' -> '.join(pretty))

In [ ]:
RANDOMISATION = 'cascading'   # change me: 'cascading' (top-down, cumulative) or 'independent' (one layer at a time)
N_LAYERS = 16                 # change me: how many layers (counting from the output) to randomise, 1..16
ABS = True                    # change me: compare |attributions| (as in the paper) or signed attributions
TARGET = pred_idx

def attribution_map(method, m):
    a = explain(method, m, x, TARGET).detach().sum(1)[0]   # sum over colour channels -> (H, W)
    return (a.abs() if ABS else a).cpu().numpy()

def similarity(a, b):
    if a.std() == 0 or b.std() == 0:   # e.g. an all-zero Grad-CAM map
        return np.nan
    return spearmanr(a.ravel(), b.ravel()).statistic

torch.manual_seed(0)
t0 = time.perf_counter()
stages = ['original'] + [layer_names[n] for n in vgg_layers[::-1][:N_LAYERS]]
maps = {meth: [attribution_map(meth, model)] for meth in SANITY_METHODS}
rand_model = copy.deepcopy(model)
for layer in vgg_layers[::-1][:N_LAYERS]:
    if RANDOMISATION == 'independent':
        rand_model = copy.deepcopy(model)                 # start again from the trained model
    dict(rand_model.named_modules())[layer].reset_parameters()   # re-initialise this layer's weights
    for meth in SANITY_METHODS:
        maps[meth].append(attribution_map(meth, rand_model))
scores = {meth: [similarity(maps[meth][0], m) for m in maps[meth]] for meth in SANITY_METHODS}
print(f'Done in {time.perf_counter() - t0:.1f}s')

In [ ]:
# Maps at some of the randomisation stages (each map normalised to its own range)
show = [s for s in ['original', 'fc8', 'fc7', 'fc6', 'conv5_3', 'conv5_1', 'conv4_1', 'conv3_1', 'conv2_1', 'conv1_1'] if s in stages]
fig, axes = plt.subplots(len(SANITY_METHODS), len(show), figsize=(1.7 * len(show), 1.8 * len(SANITY_METHODS)), squeeze=False)
for r, meth in enumerate(SANITY_METHODS):
    for c, s in enumerate(show):
        m = maps[meth][stages.index(s)]
        ax = axes[r, c]
        if ABS:
            ax.imshow(m, cmap='gray', vmin=0, vmax=max(np.percentile(m, 99.5), 1e-12))
        else:
            v = max(np.percentile(np.abs(m), 99.5), 1e-12); ax.imshow(m, cmap=RdGn, vmin=-v, vmax=v)
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0: ax.set_title(s if s == 'original' else (f'→ {s}' if RANDOMISATION == 'cascading' else f'only {s}'), fontsize=9)
        if c == 0: ax.set_ylabel(meth, fontsize=9)
fig.suptitle(f'{RANDOMISATION.capitalize()} randomisation of VGG16 weights (from the top)')
plt.tight_layout(); plt.show()

# Similarity to the original explanation
plt.figure(figsize=(9, 3.5))
for meth in SANITY_METHODS:
    plt.plot(range(len(stages)), scores[meth], 'o-', label=meth)
plt.xticks(range(len(stages)), stages, rotation=60)
plt.axhline(0, color='grey', lw=0.8)
plt.ylabel('Spearman ρ vs. original'); plt.ylim(-0.3, 1.05)
plt.xlabel(f'{RANDOMISATION} randomisation up to layer →')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Take-home messages

* Different methods answer **different questions**, and their hyperparameters (baseline, layer...) matter as much as the method.
* **Visually appealing ≠ faithful**: Guided Backprop gives beautiful maps that barely depend on the model.
* **Contrastive** explanations (*why A rather than B?*) often answer the user's real question better.